# Playwright in Python: Basics + AI/LLM App Testing

This notebook teaches:

- What Playwright is
- When to use Playwright
- How to use Playwright in Python
- Basics of Playwright Python API
- How to test AI/LLM apps with Playwright


## 1) What is Playwright?

Playwright is a browser automation and end-to-end testing framework.

With Playwright, you can:

- Open Chromium/Firefox/WebKit browsers
- Navigate web pages
- Click, type, upload files, and assert UI behavior
- Capture screenshots, traces, and videos
- Run reliable tests locally and in CI

In Python, the package is `playwright` and can be used with:

- Sync API (`playwright.sync_api`) for simple scripts
- Async API (`playwright.async_api`) for scalable concurrent tests


## 2) When should you use Playwright?

Use Playwright when you need to validate user-facing behavior in real browsers.

Great use cases:

- End-to-end UI tests (forms, login, checkout, dashboards)
- Regression tests for critical flows
- Visual checks (screenshots)
- API + UI combined validation
- AI app behavior in chat UIs (prompt send, response render, tool status)

Not ideal when:

- Pure backend/unit logic can be tested faster without browsers
- You only need quick API contract checks (use API tests first)


## 3) Setup for Python

Run once in terminal:

```bash
pip install playwright pytest-playwright
python -m playwright install
```

If you already have a `requirements.txt`, add:

- `playwright`
- `pytest-playwright` (optional, but useful for test suites)


In [ ]:
# Optional: verify Playwright import in notebook runtime
try:
    import playwright  # noqa: F401
    print("Playwright import successful")
except Exception as e:
    print("Playwright not installed in this kernel:", e)

## 4) Playwright Python basics (Sync API)

Core objects:

- `playwright` → engine entrypoint
- `browser` → browser instance
- `context` → isolated browser session
- `page` → tab
- `locator` → robust element handle for actions/assertions


In [ ]:
from playwright.sync_api import sync_playwright

def simple_browser_demo(url="https://example.com"):
    with sync_playwright() as p:
        browser = p.chromium.launch(headless=True)
        context = browser.new_context()
        page = context.new_page()
        page.goto(url, wait_until="domcontentloaded")
        print("Page title:", page.title())
        page.screenshot(path="playwright_example.png", full_page=True)
        browser.close()

# simple_browser_demo()

## 5) Locators and waits (important)

Best practices:

- Prefer semantic selectors (`get_by_role`, `get_by_label`) when possible
- Use stable IDs/data attributes for custom components
- Avoid brittle CSS selectors tied to styling
- Wait on meaningful conditions (response text appears, element visible)

Example locator patterns:

- `page.get_by_role("button", name="Send")`
- `page.locator("#query")`
- `page.locator("[data-testid='chat-message']")`


## 6) Testing AI/LLM apps with Playwright

For AI chat apps, avoid asserting exact full text (LLM output varies).

Instead, assert:

- Input is accepted and sent
- Response bubble appears
- Response is non-empty
- Response does not contain known error markers
- Optional: context/tool panels appear
- Optional: latency/time budget under threshold

This gives stable tests while still validating real behavior.

In [ ]:
import os
from playwright.sync_api import sync_playwright

def test_llm_chat_ui(url=None, prompt=None):
    url = url or os.getenv("RAG_UI_URL", "http://127.0.0.1:8000/ui")
    prompt = prompt or "Summarize the available uploaded docs in 2 bullet points."

    with sync_playwright() as p:
        browser = p.chromium.launch(headless=False)
        context = browser.new_context()
        page = context.new_page()

        page.goto(url, wait_until="domcontentloaded")

        # Basic UI sanity
        assert page.locator("h1", has_text="RAG Chatbot").first.is_visible()
        assert page.locator("#query").is_visible()
        assert page.locator("#askBtn").is_visible()

        # Send prompt
        page.locator("#query").fill(prompt)
        page.locator("#askBtn").click()

        # Wait for a new bot message
        page.wait_for_function(
            "() => document.querySelectorAll('#chat .msg.bot').length >= 2",
            timeout=30000,
        )

        bot_messages = page.locator("#chat .msg.bot")
        latest = bot_messages.nth(bot_messages.count() - 1).inner_text().strip()

        # Robust assertions for LLM variability
        assert latest, "Assistant response was empty"
        assert "error:" not in latest.lower(), f"Assistant error: {latest}"

        page.screenshot(path="llm_chat_test_result.png", full_page=True)
        browser.close()
        return latest

# response = test_llm_chat_ui()
# print(response)

## 7) How to structure AI UI test cases

Suggested scenarios:

1. **Happy path**: send prompt -> response appears
2. **No input**: Send with empty input should not crash
3. **Long prompt**: Large input still works
4. **Error handling**: backend unavailable shows friendly error
5. **Context visibility**: "Show context" toggles and renders content
6. **Performance guardrail**: response appears within N seconds

For CI stability:

- keep assertions behavior-based
- keep prompts deterministic when possible
- use retries only for known flaky network paths


## 8) Sync vs Async in Python

- Use **sync API** for simple scripts and notebook demos.
- Use **async API** if you run many browser tasks concurrently.

Async skeleton:

```python
import asyncio
from playwright.async_api import async_playwright

async def run():
    async with async_playwright() as p:
        browser = await p.chromium.launch()
        page = await browser.new_page()
        await page.goto("https://example.com")
        print(await page.title())
        await browser.close()

asyncio.run(run())
```


## 9) Quick checklist for production-grade AI UI tests

- Use stable selectors (`id` or `data-testid`)
- Add screenshots on failure
- Store test artifacts in CI
- Validate both UX and backend errors
- Keep a small smoke suite + separate deep suite
- Track response latency trends over time


## 10) Next step in this repo

You already have a runnable script at:

- `playwright/test_script.py`

Use this notebook for learning and quick experiments, and keep the script for repeatable automated checks.